# Book Recommendation Engine using KNN

freeCodeCamp Machine Learning with Python project. Run all cells in order (Runtime → Run all).

In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip -qo book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# Remove users with fewer than 200 ratings and books with fewer than 100 ratings.
# Both counts come from the full ratings table.
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

df = df_ratings[df_ratings['user'].isin(user_counts[user_counts >= 200].index)]
df = df[df['isbn'].isin(book_counts[book_counts >= 100].index)]

# Attach titles; the same title can have several ISBNs, so keep one rating per user per title
df = pd.merge(df, df_books, on='isbn')
df = df.drop_duplicates(['title', 'user'])

# Rows are books, columns are users, values are ratings (0 = not rated)
book_user = df.pivot(index='title', columns='user', values='rating').fillna(0)

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(csr_matrix(book_user.values))

print(book_user.shape)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  distances, indices = model.kneighbors(book_user.loc[[book]].values, n_neighbors=6)

  # The nearest neighbour is the book itself, so skip it and list the rest farthest first
  recommended_books = [
    [book_user.index[i], d] for i, d in zip(indices[0][1:], distances[0][1:])
  ][::-1]

  return [book, recommended_books]

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()